In [0]:
%sql
DROP TABLE IF EXISTS dim_date;

In [0]:
from pyspark.sql import functions as F
date_limits = spark.table("sales_lakehouse.silver.sales")\
  .select(
    F.min("order_date").alias("start_date"),
    F.max("order_date").alias("end_date"))\
  .collect()[0]

date_gold = spark\
  .sql(f"""SELECT explode(sequence(
        to_date('{date_limits['start_date']}'), 
        to_date('{date_limits['end_date']}'), 
        interval 1 day
    )) as current_date""")

dim_date = date_gold.select(
    F.date_format("current_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.year("current_date").alias("year"),
    F.month("current_date").alias("month"),
    F.dayofmonth("current_date").alias("day"),
    F.date_format("current_date", "MMMM").alias("month_name"),
    F.date_format("current_date", "EEEE").alias("day_name"),
    F.when(F.dayofweek("current_date").isin(1, 7), 1).otherwise(0).alias("IsWeekend"),
    F.trunc("current_date", "MM").alias("month_start_date") 
    #transforming the date to the first day of the month(15/05/2023 -> 01/05/2023) so we can join with fact_sales
)

(dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") #if we want to change any column later
    .saveAsTable("sales_lakehouse.gold.dim_date"))


In [0]:
dim_date.printSchema()